# 1. Carregar e limpar dados

## 1.1 Importações de bibliotecas e base de dados

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression


In [2]:
# Importando base de dados do campeonato brasileiro
df = pd.read_csv("data/campeonato-brasileiro-full.csv")

df.sample(5)

,ID,rodata,data,hora,mandante,visitante,formacao_mandante,formacao_visitante,tecnico_mandante,tecnico_visitante,vencedor,arena,mandante_Placar,visitante_Placar,mandante_Estado,visitante_Estado,arrecadacao
6510,6513,1,28/04/2019,16:00,Ceara,CSA,4-2-3-1,4-2-3-1,E. Alves Moreira,M. Ribeiro Cabo,Ceara,Arena Castelão,4,0,CE,AL,NaN
5427,5428,7,11/06/2016,16:00,Ponte Preta,Chapecoense,4-2-3-1,4-2-3-1,E. Baptista,A. Ferreira,Ponte Preta,Moisés Lucarelli,2,1,SP,SC,NaN
3178,3179,10,21/07/2010,19:30,Vitoria,Goias,NaN,NaN,NaN,NaN,-,Barradão,2,2,BA,GO,NaN
6471,6472,35,18/11/2018,17:01,Botafogo-RJ,Internacional,4-2-3-1,4-1-4-1,J. Mannarino,O. Hellmann,Botafogo-RJ,Estádio Nilton Santos,1,0,RJ,RS,NaN
5070,5071,9,28/06/2015,15:59,Palmeiras,Sao Paulo,4-2-3-1,4-2-3-1,M. de Oliveira Santos,J. Osorio,Palmeiras,Allianz Parque,4,0,SP,SP,NaN


## 1.2 Análise exploratória dos dados

In [3]:
# Verificando colunas do DataFrame
df.columns

Index(['ID', 'rodata', 'data', 'hora', 'mandante', 'visitante',
       'formacao_mandante', 'formacao_visitante', 'tecnico_mandante',
       'tecnico_visitante', 'vencedor', 'arena', 'mandante_Placar',
       'visitante_Placar', 'mandante_Estado', 'visitante_Estado',
       'arrecadacao'],
      dtype='str')

In [4]:
# Mantendo apenas as colunas que serão utilizadas
df = df[['rodata','data','mandante','visitante', 'mandante_Placar', 'visitante_Placar']]

In [5]:
# Contagem de linhas e colunas do DataFrame
df.shape

(9165, 6)

In [6]:
# Verificando tipos das colunas e valores nulos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9165 entries, 0 to 9164
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   rodata            9165 non-null   int64
 1   data              9165 non-null   str  
 2   mandante          9165 non-null   str  
 3   visitante         9165 non-null   str  
 4   mandante_Placar   9165 non-null   int64
 5   visitante_Placar  9165 non-null   int64
dtypes: int64(3), str(3)
memory usage: 429.7 KB


## 1.3 Limpeza e formatação

In [7]:
# Corrigindo coluna data para o tipo datetime
df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')

In [8]:
# Renomeando colunas para facilitar a manipulação dos dados
df = df.rename(columns={'rodata':'rodada', 'data':'data', 'mandante':'mandante', 'visitante':'visitante', 'mandante_Placar':'placar_mandante', 'visitante_Placar':'placar_visitante'})

In [9]:
# Filtrando apenas os jogos a partir de 2006, ano em que o campeonato brasileiro passou a ser disputado no formato atual
df = df[df['data'] >= '2006-01-01']

In [10]:
# Nova quantidade de linhas do DataFrame após o filtro
df.shape

(7599, 6)

In [11]:
df.head()

,rodada,data,mandante,visitante,placar_mandante,placar_visitante
1566,1,2006-04-15,Juventude,Parana,1,0
1567,1,2006-04-15,Vasco,Internacional,1,1
1568,1,2006-04-16,Sao Paulo,Flamengo,1,0
1569,1,2006-04-16,Gremio,Corinthians,2,0
1570,1,2006-04-16,Botafogo-RJ,Fortaleza,1,0


In [12]:
# Arrumando o índice do DataFrame após o filtro
df = df.sort_values(by=['data', 'rodada'], ascending=[True, True]).reset_index(drop=True)

In [13]:
# Verificando o período de tempo dos dados
print(df['data'].min())
print(df['data'].max())

2006-04-15 00:00:00
2025-12-07 00:00:00


## 1.4 Criação de novas colunas

In [14]:
# Criando a coluna 'ano'
df['ano'] = df['data'].dt.year

In [15]:
# Contando quantos jogos tiveram no ano. Metade dos jogos de 2020 foram realizados em 2021
df.groupby('ano')['rodada'].count()

ano
2006    380
2007    380
2008    380
2009    380
2010    380
2011    380
2012    380
2013    380
2014    380
2015    380
2016    379
2017    380
2018    380
2019    380
2020    268
2021    492
2022    380
2023    380
2024    380
2025    380
Name: rodada, dtype: int64

In [16]:
# Entendendo quando começou o Brasileirão 2021 de fato
df[(df['ano'] == 2021) & (df['rodada'] == 1)]  

,rodada,data,mandante,visitante,placar_mandante,placar_visitante,ano
5635,1,2021-01-26,Palmeiras,Vasco,1,1,2021
5699,1,2021-05-29,Cuiaba,Juventude,2,2,2021
5700,1,2021-05-29,Bahia,Santos,3,0,2021
5701,1,2021-05-29,Sao Paulo,Fluminense,0,0,2021
5702,1,2021-05-30,Atletico-MG,Fortaleza,1,2,2021
5703,1,2021-05-30,Ceara,Gremio,3,2,2021
5704,1,2021-05-30,Flamengo,Palmeiras,1,0,2021
5705,1,2021-05-30,Chapecoense,Bragantino,0,3,2021
5706,1,2021-05-30,Corinthians,Atletico-GO,0,1,2021
5707,1,2021-05-30,Athletico-PR,America-MG,1,0,2021


In [17]:
# Alterando para que a coluna de ano mostre em qual temporada o ocorreu o jogo
df['ano'] = np.where(
    (df['ano'] == 2021) & (df['data'] < '2021-04-01'),
    2020,
    df['ano']
)

In [18]:
# Condições pontuação do time mandante
condicoes_pontuacao_mandante = [
    df['placar_mandante'] > df['placar_visitante'],
    df['placar_mandante'] == df['placar_visitante'],
    df['placar_visitante'] > df['placar_mandante']
]

# Quantidade de pontos baseado na condição
pontuacao = [3, 1, 0]

# Adição da coluna com as pontuações calculadas
df['pontos_mandante'] = np.select(condicoes_pontuacao_mandante, pontuacao)


In [19]:
# Condições pontuação do time visitante
condicoes_pontuacao_visitante = [
    df['placar_visitante'] > df['placar_mandante'],
    df['placar_mandante'] == df['placar_visitante'],
    df['placar_mandante'] > df['placar_visitante']
]

# Quantidade de pontos baseado na condição
pontuacao = [3, 1, 0]

# Adição da coluna com as pontuações calculadas
df['pontos_visitante'] = np.select(condicoes_pontuacao_visitante, pontuacao)


In [20]:
condicoes_resultado = [
    df['placar_mandante'] > df['placar_visitante'],
    df['placar_mandante'] == df['placar_visitante'],
    df['placar_mandante'] < df['placar_visitante']
]

resultados = ['M', 'E', 'V']

df['resultado'] = np.select(condicoes_resultado, resultados, default='Indefinido')

In [21]:
condicoes_resultado = [
    df['placar_mandante'] > df['placar_visitante'],
    df['placar_mandante'] == df['placar_visitante'],
    df['placar_mandante'] < df['placar_visitante']
]

resultados = [1, 0.5, 0]

df['resultado_elo_mandante'] = np.select(condicoes_resultado, resultados)
df['resultado_elo_visitante'] = (1-(np.select(condicoes_resultado, resultados)))

In [22]:
df.sample(5)

,rodada,data,mandante,visitante,placar_mandante,placar_visitante,ano,pontos_mandante,pontos_visitante,resultado,resultado_elo_mandante,resultado_elo_visitante
4853,30,2018-10-21,Fluminense,Atletico-MG,1,0,2018,3,0,M,1.0,0.0
2333,6,2012-06-24,Corinthians,Palmeiras,2,1,2012,3,0,M,1.0,0.0
1047,29,2008-10-11,Portuguesa,Coritiba,0,0,2008,1,1,E,0.5,0.5
5074,14,2019-08-11,Avai,Cruzeiro,2,2,2019,1,1,E,0.5,0.5
3806,1,2016-05-15,America-MG,Fluminense,0,1,2016,0,3,V,0.0,1.0


# 2. Classificação de uma temporada específica (teste de sanidade)

## 2.1 Criando dataframes do time mandante

In [23]:
# Criando dataframe apenas com os jogos dos times mandante
df_mandante = df[['rodada', 'data', 'ano', 'mandante', 'placar_mandante', 'placar_visitante', 'pontos_mandante', 'visitante']]

# Renomeando as colunas
df_mandante = df_mandante.rename(columns={'rodada':'rodada', 'data':'data', 'ano':'ano', 'mandante':'time', 'placar_mandante':'gols_pro', 'placar_visitante':'gols_contra', 'pontos_mandante':'pontos', 'visitante':'adversario'})

In [24]:
condicoes_resultado_mandante = [
    df_mandante['gols_pro'] > df_mandante['gols_contra'],
    df_mandante['gols_pro'] == df_mandante['gols_contra'],
    df_mandante['gols_pro'] < df_mandante['gols_contra']
]

resultado_mandante = ['M', 'E', 'V']

In [25]:
df_mandante['mando'] = 1

df_mandante['resultado'] = np.select(condicoes_resultado_mandante, resultado_mandante, default='Indefinido')

df_mandante.head()

,rodada,data,ano,time,gols_pro,gols_contra,pontos,adversario,mando,resultado
0,1,2006-04-15,2006,Juventude,1,0,3,Parana,1,M
1,1,2006-04-15,2006,Vasco,1,1,1,Internacional,1,E
2,1,2006-04-16,2006,Sao Paulo,1,0,3,Flamengo,1,M
3,1,2006-04-16,2006,Gremio,2,0,3,Corinthians,1,M
4,1,2006-04-16,2006,Botafogo-RJ,1,0,3,Fortaleza,1,M


## 2.2 Criando dataframe com jogos do time visitante

In [26]:
# Criando dataframe apenas com os jogos dos times visitantes
df_visitante = df[['rodada', 'data', 'ano', 'visitante', 'placar_visitante', 'placar_mandante', 'pontos_visitante', 'mandante']]

# Renomeando as colunas
df_visitante = df_visitante.rename(columns={'rodada':'rodada', 'data':'data', 'ano':'ano', 'visitante':'time', 'placar_visitante':'gols_pro', 'placar_mandante':'gols_contra', 'pontos_visitante':'pontos', 'mandante':'adversario'})

In [27]:
condicoes_resultado_visitante = [
    df_visitante['gols_pro'] > df_visitante['gols_contra'],
    df_visitante['gols_pro'] == df_visitante['gols_contra'],
    df_visitante['gols_pro'] < df_visitante['gols_contra']
]

resultado_visitante = ['V', 'E', 'M']

In [28]:
df_visitante['mando'] = 0

df_visitante['resultado'] = np.select(condicoes_resultado_visitante, resultado_visitante, default='Indefinido')

df_visitante.head()

,rodada,data,ano,time,gols_pro,gols_contra,pontos,adversario,mando,resultado
0,1,2006-04-15,2006,Parana,0,1,0,Juventude,0,M
1,1,2006-04-15,2006,Internacional,1,1,1,Vasco,0,E
2,1,2006-04-16,2006,Flamengo,0,1,0,Sao Paulo,0,M
3,1,2006-04-16,2006,Corinthians,0,2,0,Gremio,0,M
4,1,2006-04-16,2006,Fortaleza,0,1,0,Botafogo-RJ,0,M


## 2.3 Juntando dataframes de mandante e visitante em um

In [29]:
# Juntando os dois dataframes em um único
df_tabela_total = pd.concat([df_mandante, df_visitante])
df_tabela_total.sample(5)

,rodada,data,ano,time,gols_pro,gols_contra,pontos,adversario,mando,resultado
483,11,2007-07-14,2007,Gremio,1,1,1,Palmeiras,1,E
7581,34,2025-12-03,2025,Vitoria,0,4,0,Bragantino,0,M
629,26,2007-09-15,2007,Nautico,3,0,3,Goias,0,V
3787,37,2015-11-29,2015,Chapecoense,1,3,0,Goias,1,V
3532,12,2015-07-08,2015,Coritiba,0,0,1,Ponte Preta,1,E


## 2.4 Criando tabela de uma temporada específica a partir dos dados

In [30]:
# Filtrando os dados para o ano de 2024
df_2024 = df_tabela_total[df_tabela_total['ano'] == 2024]

# Agrupando por time e somando APENAS gols_pro, gols_contra e pontos e ordenando por pontos
df_ranking = df_2024.groupby('time')[['gols_pro', 'gols_contra', 'pontos']].sum().sort_values(by='pontos', ascending=False)

df_ranking

,gols_pro,gols_contra,pontos
time,,,
Botafogo-RJ,59,29,79
Palmeiras,60,33,73
Flamengo,61,42,70
Fortaleza,53,39,68
Internacional,53,36,65
Sao Paulo,53,43,59
Corinthians,54,45,56
Bahia,49,49,53
Cruzeiro,43,41,52


In [31]:
# Checa se a coluna 'data' está ordenada
df['data'].is_monotonic_increasing

True

# 3. Cálculo de elo dos times

## 3.1 Resultado esperado de cada jogo sendo mandante (M: Mandante venceu, E: Empate, V: Visitante venceu)

In [32]:
df.value_counts('resultado', normalize=True) *100

resultado
M    49.085406
E    26.792999
V    24.121595
Name: proportion, dtype: float64

## 3.2 Funções para calcular Elo de cada time

In [33]:
def prob_esperada_m(mandante, visitante, h):
    return 1/(1 + 10**((visitante - (mandante + h))/400))

In [34]:
def prob_esperada_v(visitante, mandante, h):
    return 1/(1 + 10**(((mandante + h) - visitante )/400))

In [35]:
def atualizar_elo_m(mandante, k, resultado, probabilidade_m):
    return mandante + k * (resultado-probabilidade_m)

In [36]:
def atualizar_elo_v(visitante, k, resultado, probabilidade_v):
    return visitante + k * (resultado - probabilidade_v)

## 3.3 Loop de cálculo do elo em cada jogo

In [37]:
h = 100
k = 10

In [38]:
# Criando dicionário que guarda o elo de cada time
elo = {}

# Listas que salvam o elo do time antes de uma partida, para ser atualizado
elo_mandante_antes = []
elo_visitante_antes = []

# Loop de atualização do elo de cada time
for jogo in df.itertuples():

    # Busca o elo do time selecionado, caso o time ainda não esteja salvo, atribui elo 1500
    elo_m = elo.get(jogo.mandante, 1500)
    elo_v = elo.get(jogo.visitante, 1500)

    # Salva o elo do time antes da partida
    elo_mandante_antes.append(elo_m)
    elo_visitante_antes.append(elo_v)

    # Usa a função que calcula a probabilidade do time vencer a partida baseada no elo, considerando a vantagem do mandante (h)
    prob_m = prob_esperada_m(elo_m, elo_v, h)
    prob_v = prob_esperada_v(elo_v, elo_m, h)
    
    # Usa a função que atualiza o elo da equipe baseado no resultado real da partida
    novo_elo_m = atualizar_elo_m(elo_m, k, jogo.resultado_elo_mandante, prob_m)
    novo_elo_v = atualizar_elo_v(elo_v, k, jogo.resultado_elo_visitante, prob_v)

    # Atualiza o dicionário com o novo elo do time
    elo[jogo.mandante] = novo_elo_m
    elo[jogo.visitante] = novo_elo_v

In [39]:
# Cria coluna com o elo de cada time em cada partida jogada, desde quando todos tinham 1500 de elo até o último
df['elo_mandante_antes'] = elo_mandante_antes
df['elo_visitante_antes'] = elo_visitante_antes

In [40]:
# Teste para saber se os elos estão batendo
print(sum(elo.values()))
print(1500*len(elo))

66000.0
66000


# 4. Modelo "burro" (Baseado apenas no mando de campo)

## 4.1 Coluna de parâmetro do modelo

In [41]:
# Criando coluna com a diferença de elo de cada time naquele determinado jogo
df['diff_elo'] = df.elo_mandante_antes - df.elo_visitante_antes

In [42]:
df.sample(5)

,rodada,data,mandante,visitante,placar_mandante,placar_visitante,ano,pontos_mandante,pontos_visitante,resultado,resultado_elo_mandante,resultado_elo_visitante,elo_mandante_antes,elo_visitante_antes,diff_elo
6892,7,2024-06-01,Gremio,Bragantino,0,2,2024,0,3,V,0.0,1.0,1591.025542,1563.183513,27.842029
3501,9,2015-06-27,Chapecoense,Sport,1,1,2015,1,1,E,0.5,0.5,1496.953757,1514.667541,-17.713784
4644,9,2018-06-03,Santos,Vitoria,5,2,2018,3,0,M,1.0,0.0,1582.501105,1481.557317,100.943788
2501,23,2012-09-08,Corinthians,Gremio,3,1,2012,3,0,M,1.0,0.0,1570.136072,1580.649490,-10.513418
7458,25,2025-09-28,Bragantino,Santos,2,2,2025,1,1,E,0.5,0.5,1528.562656,1516.849412,11.713244


## 4.2 Separação de treino, validação e teste

In [43]:
# Retirando dados de 2006 a 2008 (aquecimento do modelo)
df_modelo = df[df['ano']>2008]

In [44]:
# Criando dataframe de treino
df_treino = df_modelo[df_modelo['ano']<2023]

In [45]:
# Criando dataframe de validação
df_validacao = df_modelo[df_modelo['ano'] == 2023]

In [46]:
# Criando dataframe de teste
df_teste = df_modelo[df_modelo['ano']>2023]

## 4.3 Log loss dos resultados reais comparado ao padrão esperado de vitória

In [47]:
freq_treino = list(df_treino.value_counts('resultado', normalize=True).sort_index())

In [48]:
prob_burro = np.tile(freq_treino, (len(df_validacao), 1))

In [49]:
log_loss(df_validacao.resultado, prob_burro)

1.0625208892303566

# 5. Regressão logística

In [50]:
modelo = LogisticRegression()
modelo.fit(df_treino[['diff_elo']], df_treino['resultado'])

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [51]:
prob_modelo = modelo.predict_proba(df_validacao[['diff_elo']])

In [52]:
log_loss(df_validacao.resultado, prob_modelo)

1.0338306477095958

In [53]:
# Avaliação em várias temporadas

lista_prob_temporada = []
lista_resultado = []

for ano_teste in [2021, 2022, 2023]:
    treino = df_modelo[df_modelo.ano < ano_teste] 
    teste = df_modelo[df_modelo.ano == ano_teste]
    modelo_teste = LogisticRegression()
    modelo_teste.fit(treino[['diff_elo']], treino['resultado'])
    prob_temporada = modelo_teste.predict_proba(teste[['diff_elo']])
    lista_prob_temporada.append(prob_temporada)
    lista_resultado.append(teste.resultado)
    print(log_loss(teste.resultado,prob_temporada, labels=['E','M','V']))




1.0458063099777957
1.0326250449365304
1.0338306477095958
